# Stage 7: Evaluation & Interpretability
**Environment:** Kaggle Notebook (GPU T4)  
**Requires:** `pip install pykan`

In [ ]:
!pip install pykan -q

In [ ]:
import numpy as np, pandas as pd, cv2, os, glob, json
from typing import Dict, List
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from kan import KAN
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, roc_curve, classification_report, confusion_matrix
import matplotlib.pyplot as plt, matplotlib.gridspec as gridspec, seaborn as sns
from tqdm.notebook import tqdm

%matplotlib inline
plt.rcParams['figure.dpi'] = 120; sns.set_style('whitegrid')

INPUT_DIR = '/kaggle/input/artifact-dataset'
OUTPUT_DIR = '/kaggle/working'
CACHE_DIR = os.path.join(OUTPUT_DIR, 'phase_cache')
MODEL_DIR = os.path.join(OUTPUT_DIR, 'models')
ABLATION_DIR = os.path.join(MODEL_DIR, 'ablations')
REPORT_DIR = os.path.join(MODEL_DIR, 'final_report')
os.makedirs(REPORT_DIR, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load results
all_results = {}
for fname in ['all_results.json','results_kan.json','results_b3.json']:
    p = os.path.join(MODEL_DIR, fname)
    if os.path.exists(p):
        with open(p) as f: d = json.load(f)
        if isinstance(d,dict) and 'model' in d: all_results[d['model']] = d
        else: all_results.update(d)
        print(f'Loaded {fname}')

ablation_data = {}
for af in glob.glob(os.path.join(ABLATION_DIR,'*.csv')):
    ablation_data[os.path.splitext(os.path.basename(af))[0]] = pd.read_csv(af)
print(f'Models: {len(all_results)}, Ablations: {list(ablation_data.keys())}')

In [ ]:
# Cell 2: Comprehensive Comparison
rows = []
for k,r in all_results.items():
    name = r.get('model',k)
    if 'RGB' in name: domain='Spatial'
    elif 'Mag' in name: domain='Freq-Mag'
    else: domain='Freq-Phase'
    arch = 'KAN' if 'KAN' in name else ('ViT' if 'ViT' in name else ('ResNet' if 'Res' in name else 'MLP'))
    rows.append({'Model':name,'Arch':arch,'Domain':domain,'AUC':r.get('test_auc',0),
                 'Accuracy':r.get('test_accuracy',0),'Params':r.get('n_parameters',0)})
comp_df = pd.DataFrame(rows).sort_values('AUC',ascending=False)
print('='*70 + '\nFINAL COMPARISON\n' + '='*70)
print(comp_df.to_string(index=False))

fig = plt.figure(figsize=(18,10))
gs = gridspec.GridSpec(2,3,figure=fig,hspace=0.35,wspace=0.3)
fig.suptitle('KAN-Phase Deepfake Detection: Final Results', fontsize=16, fontweight='bold', y=0.98)

# AUC bars
ax1 = fig.add_subplot(gs[0,0])
pal = {'KAN':'#e74c3c','MLP':'#3498db','ResNet':'#2ecc71','ViT':'#9b59b6'}
colors = [pal.get(r['Arch'],'#95a5a6') for _,r in comp_df.iterrows()]
ax1.barh(comp_df['Model'], comp_df['AUC'], color=colors)
ax1.set_xlabel('AUC'); ax1.set_xlim(0,1.05); ax1.set_title('Test AUC')

# Param efficiency
ax2 = fig.add_subplot(gs[0,1])
for _,r in comp_df.iterrows():
    ax2.scatter(r['Params'],r['AUC'],s=120,c=pal.get(r['Arch'],'gray'),edgecolors='k',lw=0.5,zorder=5)
    ax2.annotate(r['Model'][:15],(r['Params'],r['AUC']),fontsize=7,xytext=(5,5),textcoords='offset points')
ax2.set_xlabel('Params'); ax2.set_ylabel('AUC'); ax2.set_xscale('log'); ax2.set_title('Efficiency')

# Domain comparison
ax3 = fig.add_subplot(gs[0,2])
comp_df.groupby('Domain')['AUC'].mean().plot(kind='bar',ax=ax3,color=['steelblue','coral','gray'])
ax3.set_ylabel('Mean AUC'); ax3.set_title('Domain Comparison'); ax3.set_xticklabels(ax3.get_xticklabels(),rotation=0)

# A4
ax4 = fig.add_subplot(gs[1,0])
if 'ablation_a4' in ablation_data:
    a4 = ablation_data['ablation_a4'].sort_values('AUC')
    ax4.barh(a4['Arch'], a4['AUC'], color='teal')
ax4.set_xlabel('AUC'); ax4.set_title('A4: Architecture Sweep')

# A5
ax5 = fig.add_subplot(gs[1,1])
if 'ablation_a5_jpeg' in ablation_data:
    j = ablation_data['ablation_a5_jpeg']
    ax5.plot(j['Quality'],j['AUC'],'o-',lw=2,color='steelblue',label='JPEG')
if 'ablation_a5_blur' in ablation_data:
    b = ablation_data['ablation_a5_blur']
    ax5b = ax5.twiny()
    ax5b.plot(b['Kernel'],b['AUC'],'s--',lw=2,color='coral',label='Blur')
    ax5b.set_xlabel('Blur Kernel',color='coral',fontsize=8)
ax5.set_xlabel('JPEG Q'); ax5.set_ylabel('AUC'); ax5.set_ylim(0,1.05); ax5.set_title('A5: Robustness')

# A6
ax6 = fig.add_subplot(gs[1,2])
if 'ablation_a6_loo' in ablation_data:
    loo = ablation_data['ablation_a6_loo'].sort_values('OOD_AUC')
    ax6.barh(loo['Generator'], loo['OOD_AUC'], color=['#2ecc71' if a>0.7 else '#e74c3c' for a in loo['OOD_AUC']])
    ax6.axvline(x=0.5,color='k',ls='--',alpha=0.5)
ax6.set_title('A6: OOD Generalisation')
fig.savefig(os.path.join(REPORT_DIR,'final_overview.png'),dpi=150,bbox_inches='tight')
plt.show()
comp_df.to_csv(os.path.join(REPORT_DIR,'final_comparison.csv'),index=False)

In [ ]:
# Cell 3: Per-Generator AUC & KAN Interpretability
cache_files = sorted(glob.glob(os.path.join(CACHE_DIR, 'phase_maps_*.npy')))
phase_results = np.load(cache_files[-1], allow_pickle=True).item()
meta_files = glob.glob(os.path.join(INPUT_DIR, '**', 'metadata.csv'), recursive=True)
metadata_df = pd.concat([pd.read_csv(mf).assign(generator=os.path.basename(os.path.dirname(mf)))
                         for mf in meta_files], ignore_index=True) if meta_files else None

real_gens = set()
for gen in phase_results:
    if metadata_df is not None:
        gdf = metadata_df[metadata_df['generator']==gen]
        if len(gdf)>0 and gdf['target'].mode().iloc[0]==0: real_gens.add(gen)
    elif 'real' in gen.lower(): real_gens.add(gen)

X_all, y_all = [], []
for gen,maps in phase_results.items():
    for m in maps: X_all.append(m.flatten()); y_all.append(0 if gen in real_gens else 1)
X_all = np.array(X_all, dtype=np.float32); y_all = np.array(y_all)
scaler = StandardScaler(); pca = PCA(n_components=128, random_state=42)
X_pca = pca.fit_transform(scaler.fit_transform(X_all)).astype(np.float32)

X_tv,X_te,y_tv,y_te = train_test_split(X_pca,y_all,test_size=0.2,stratify=y_all,random_state=42)
X_tr,X_va,y_tr,y_va = train_test_split(X_tv,y_tv,test_size=0.15,stratify=y_tv,random_state=42)

kan = KAN(width=[128,64,32,1], grid=5, k=3, device=str(DEVICE))
crit = nn.BCEWithLogitsLoss(); opt = torch.optim.Adam(kan.parameters(), lr=1e-3, weight_decay=1e-4)
loader = DataLoader(TensorDataset(torch.FloatTensor(X_tr), torch.LongTensor(y_tr)), batch_size=64, shuffle=True)
for _ in tqdm(range(30), desc='Training KAN'):
    kan.train()
    for xb,yb in loader:
        xb,yb=xb.to(DEVICE),yb.float().to(DEVICE)
        opt.zero_grad(); crit(kan(xb).squeeze(-1),yb).backward(); opt.step()

# Per-generator AUC
real_feats = []
for g in real_gens:
    for m in phase_results[g][:50]:
        real_feats.append(pca.transform(scaler.transform(m.flatten().reshape(1,-1))).astype(np.float32)[0])
real_feats = np.array(real_feats)

per_gen = []
kan.eval()
for gen in sorted(phase_results.keys()):
    if gen in real_gens: continue
    fake_f = np.array([pca.transform(scaler.transform(m.flatten().reshape(1,-1))).astype(np.float32)[0]
                       for m in phase_results[gen]])
    Xe = np.vstack([real_feats, fake_f]); ye = np.array([0]*len(real_feats)+[1]*len(fake_f))
    with torch.no_grad():
        p = torch.sigmoid(kan(torch.FloatTensor(Xe).to(DEVICE)).squeeze(-1)).cpu().numpy()
    per_gen.append({'Generator':gen,'AUC':roc_auc_score(ye,p),'N':len(fake_f)})

pg_df = pd.DataFrame(per_gen).sort_values('AUC',ascending=False)
print('Per-Generator AUC:'); print(pg_df.to_string(index=False))
if len(pg_df)>0:
    fig,ax = plt.subplots(figsize=(10,max(4,len(pg_df)*0.4)))
    ax.barh(pg_df['Generator'], pg_df['AUC'],
            color=['#2ecc71' if a>0.8 else '#f39c12' if a>0.6 else '#e74c3c' for a in pg_df['AUC']])
    ax.set_xlabel('AUC'); ax.set_title('Per-Generator Detection AUC'); ax.invert_yaxis()
    plt.tight_layout(); fig.savefig(os.path.join(REPORT_DIR,'per_gen_auc.png'),dpi=150); plt.show()
pg_df.to_csv(os.path.join(REPORT_DIR,'per_generator_auc.csv'),index=False)

In [ ]:
# Cell 4: KAN Interpretability
print('='*50 + '\nKAN INTERPRETABILITY\n' + '='*50)

# Feature importance via gradients
X_grad = torch.FloatTensor(X_tr[:30]).to(DEVICE).requires_grad_(True)
output = kan(X_grad).sum()
output.backward()

if X_grad.grad is not None:
    importance = X_grad.grad.abs().mean(dim=0).cpu().numpy()
    fig,(ax1,ax2) = plt.subplots(1,2,figsize=(14,5))
    fig.suptitle('KAN-Phase Feature Importance', fontweight='bold')
    ax1.bar(range(len(importance)), importance, color='steelblue', alpha=0.7)
    ax1.set_xlabel('PCA Component'); ax1.set_ylabel('Mean |Gradient|'); ax1.set_title('All Components')
    top_k = 20; top_idx = np.argsort(importance)[-top_k:][::-1]
    ax2.barh(range(top_k), importance[top_idx], color='coral')
    ax2.set_yticks(range(top_k)); ax2.set_yticklabels([f'PC-{i}' for i in top_idx])
    ax2.set_xlabel('Mean |Gradient|'); ax2.set_title(f'Top {top_k} Components'); ax2.invert_yaxis()
    plt.tight_layout(); fig.savefig(os.path.join(REPORT_DIR,'feature_importance.png'),dpi=150); plt.show()
    
    # Phase attention heatmap
    components = pca.components_[:len(importance)]
    weighted = np.zeros(components.shape[1])
    for i in range(len(importance)): weighted += importance[i] * np.abs(components[i])
    size = int(np.sqrt(len(weighted)))
    heatmap = weighted.reshape(size,size)
    heatmap = (heatmap-heatmap.min())/(heatmap.max()-heatmap.min()+1e-8)
    
    # Sample images
    real_gen = list(real_gens)[0] if real_gens else None
    fake_gen = [g for g in phase_results if g not in real_gens]
    samples = []
    if real_gen: samples.append(('Real', phase_results[real_gen][0]))
    if fake_gen: samples.append((f'Fake ({fake_gen[0][:12]})', phase_results[fake_gen[0]][0]))
    
    if samples:
        fig,axes = plt.subplots(2,len(samples),figsize=(6*len(samples),10))
        if len(samples)==1: axes = axes.reshape(-1,1)
        fig.suptitle('Phase Attention Heatmaps', fontweight='bold')
        for j,(label,pm) in enumerate(samples):
            axes[0,j].imshow(pm, cmap='twilight', vmin=0, vmax=1); axes[0,j].set_title(f'{label} Phase'); axes[0,j].axis('off')
            im = axes[1,j].imshow(heatmap, cmap='hot'); axes[1,j].set_title(f'{label} Attention'); axes[1,j].axis('off')
            fig.colorbar(im, ax=axes[1,j], fraction=0.046, pad=0.04)
        plt.tight_layout(); fig.savefig(os.path.join(REPORT_DIR,'attention_heatmaps.png'),dpi=150); plt.show()

# pykan network visualisation
try:
    kan(torch.FloatTensor(X_tr[:5]).to(DEVICE))
    kan.plot(beta=3); plt.savefig(os.path.join(REPORT_DIR,'kan_network.png'),dpi=150); plt.show()
except Exception as e: print(f'pykan plot: {e}')

In [ ]:
# Cell 5: Final Summary
print('\n' + '='*70)
print('FINAL SUMMARY REPORT')
print('KAN-Driven Phase-Spectrum Analysis for Deepfake Detection')
print('='*70)
print('\n1. MODEL COMPARISON')
print(comp_df[['Model','AUC','Accuracy','Params']].to_string(index=False))
if len(comp_df)>0:
    best = comp_df.iloc[0]
    print(f'\n   Best: {best["Model"]} (AUC={best["AUC"]:.4f})')
print('\n2. KEY FINDINGS')
print('   - Phase spectrum provides discriminative features for deepfake detection')
print('   - KAN learns interpretable B-spline activations on frequency features')
if 'ablation_a4' in ablation_data:
    ba = ablation_data['ablation_a4'].iloc[0]
    print(f'   - Best KAN arch: {ba["Arch"]} (AUC={ba["AUC"]:.4f})')
if 'ablation_a6_loo' in ablation_data:
    lo = ablation_data['ablation_a6_loo']
    print(f'   - OOD generalisation: mean AUC={lo["OOD_AUC"].mean():.4f}')
print('\n3. SAVED OUTPUTS')
for f in sorted(os.listdir(REPORT_DIR)): print(f'   {f}')
print('\n' + '='*70 + '\nEND OF ANALYSIS\n' + '='*70)